**Null customer IDs**

In [5]:
from pyspark.sql.functions import current_timestamp

null_customers = spark.sql("""
SELECT COUNT(*) AS cnt
FROM silver.customers
WHERE customer_id IS NULL
""").collect()[0]["cnt"]


StatementMeta(, d63d8cf4-798c-4630-acf8-9a9a243f6279, 7, Finished, Available, Finished, False)

**Duplicate order IDs**

In [6]:
duplicate_orders = spark.sql("""
SELECT COUNT(*) AS cnt
FROM (
    SELECT order_id
    FROM silver.orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
""").collect()[0]["cnt"]

StatementMeta(, d63d8cf4-798c-4630-acf8-9a9a243f6279, 8, Finished, Available, Finished, False)

**Negative order item price**

In [7]:
negative_prices = spark.sql("""
SELECT COUNT(*) AS cnt
FROM silver.order_items
WHERE price < 0
""").collect()[0]["cnt"]

StatementMeta(, d63d8cf4-798c-4630-acf8-9a9a243f6279, 9, Finished, Available, Finished, False)

**Final Validation**

In [8]:
validation_results = [
    ("Null Customer IDs", null_customers),
    ("Duplicate Orders", duplicate_orders),
    ("Negative Prices", negative_prices)
]

validation_df = spark.createDataFrame(
    validation_results,
    ["validation_check", "failed_records"]
).withColumn("validation_timestamp", current_timestamp())

display(validation_df)


StatementMeta(, d63d8cf4-798c-4630-acf8-9a9a243f6279, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb35716d-9167-408f-a970-df1fa0b1efdb)

In [9]:
validation_df.write.mode("overwrite").format("delta").saveAsTable("gold.validation_results")

StatementMeta(, d63d8cf4-798c-4630-acf8-9a9a243f6279, 11, Finished, Available, Finished, False)